# Manual exploration of the datasets created by gpt5.2 and gemini

### init

In [1]:
import sys
from datasets import load_dataset,concatenate_datasets
new_path = r'/home/creux/Documents/AI/VTikz/VariabilityBenchmark'
sys.path.append(new_path)

ds1 = load_dataset("CharlyR/vtikz-evaluation", "simpleLLM_benchmark_openaigpt5.2_pk_1_t_0.7_v1.63",split="tikz")
ds2 = load_dataset("CharlyR/vtikz-evaluation", "simpleLLM_benchmark_googlegemini3propreview_pk_1_t_0.7_v1.63",split="tikz")
ds = concatenate_datasets([ds1,ds2])
df =ds.to_pandas()


/home/creux/miniconda3/envs/varbench/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
any(metric==100 for metric in df["TemplateMetric"][0][0])

True

In [3]:
import numpy as np
df = df[df["TemplateMetric"].apply(
    lambda metrics: all(len(metric) > 0 and metric[0] != 100 for metric in metrics)
)]

In [ ]:
from IPython.display import HTML, display
import base64

def display_images_row(*img_jsons, max_height=500, margin=10):
    imgs = []
    for img in img_jsons:
        if img is None:
            continue
        b64 = base64.b64encode(img["bytes"]).decode("utf-8")
        imgs.append(
            f"<img src='data:image/png;base64,{b64}' "
            f"style='max-height:{max_height}px; margin-right:{margin}px;'>"
        )
    html = "<div style='display:flex; align-items:center;'>" + "".join(imgs) + "</div>"
    display(HTML(html))



In [5]:
import difflib

def single_patch(input: str, prediction: str) -> str:
    """Generates a patch from the given input(which will be splitted) and the prediction"""
    solution_split = prediction.splitlines()
    input_split = input.splitlines()
    current_diff = "\n".join(
        list(difflib.unified_diff(input_split, solution_split, n=0))[2:]
    )
    return current_diff


### Explore

In [11]:
i=  0
hard_items =[]
i-=  1

In [140]:
len(df)

81

In [141]:
i+=1
row = df.iloc[i]
print(row["instruction"])
print(row["id"])
display_images_row(
    row["image_input"],
    row["image_solution"][0],
    row["images_result"][0],
)
open("pred.tex","w").write(row["predictions"][0])
d1 =single_patch(row["code"],row["predictions"][0])
print(d1)

IndexError: single positional indexer is out-of-bounds

In [103]:
hard_items.append((row["id"],row["modification_type"],row["CrystalBleuPatchMetric"],row["LineMetric"]))

In [137]:
hard_items_fi = [(i,ci) for (i,ci,_,_) in hard_items]

In [ ]:
import json
pt = "/home/creux/Documents/AI/VTikz/VariabilityBenchmark/notebooks/manual_exploration/hard_ids.json"
json.dump(list(hard_items_fi),open(pt,"w"))

In [78]:
d2=single_patch(row["code"],row["template_solution_code"][0])
print(d2)


@@ -29 +29,4 @@

-\draw [fill=red!60] (\d, 0) rectangle ({\d+\w}, \h);
+\fill [fill = red!50] (\d,0) rectangle ({\d+\w}, {\h});
+\fill [fill = red!30] (\d, {0.2*\h}) rectangle ({\d+\w}, {\h});
+\fill [fill = red!10] (\d, {0.6*\h}) rectangle ({\d+\w}, \h);
+\draw [fill opacity=0] (\d, 0) rectangle ({\d+\w}, \h);


In [79]:
from vtikz.evaluation.template import template_valid,create_default
from vtikz.dataset_workflow.utils import uncomment_code,unify_code
path = "/home/creux/Documents/AI/VTikz/VariabilityBenchmark/dataset/tikz/benchmark/signal_colored/solutions/solution6.tex"

th_fil = unify_code(uncomment_code((open(path).read())))
template_valid(th_fil,row["predictions"][0])

True

In [56]:
print(single_patch(row["predictions"][0],th_fil))

@@ -26,2 +26,2 @@

-\fill [Brown100] (0,0) arc (0:90:32 and 96) arc (90:0:96);
-\fill [Brown200] (32,0) arc (0:90:64 and 96) arc (90:0:96);
+\fill [BlueGrey100] (0,0) arc (0:90:32 and 96) arc (90:0:96);
+\fill [BlueGrey200] (32,0) arc (0:90:64 and 96) arc (90:0:96);
@@ -30 +30 @@

-\fill \ifnum\j=0 \ifnum \i=-1 [Brown700] \else [Brown800] \fi\else [Brown900] \fi
+\fill \ifnum\j=0 \ifnum \i=-1 [Brown§choice([400,500,600],600)] \else [Brown§choice([500,600,700],700)] \fi\else [Brown§choice([600,700,800],800)] \fi
@@ -37 +37 @@

-\reflect[split={Brown700 and Brown800}]{
+\reflect[split={Brown§choice([400,500,600,700,800],600) and Brown§choice([500,600,700,800,900],700)}]{
@@ -42,0 +43,3 @@

+\fill [Grey50] (§rangei(-40,15),§rangei(10,20)) circle [radius=§rangei(15,10)];
+\fill [Grey50] (§rangei(-60,15),§rangei(-30,20)) circle [radius=§rangei(15,10)];
+\fill [Grey50] (§rangei(-80,15),§rangei(10,20)) circle [radius=§rangei(15,10)];
@@ -60 +63 @@

-\fill [Brown900]
+\fill [BlueGrey900]
@@ -6

In [124]:

[open(f"solution{i}.tex","w").write(template) for i,template in enumerate(row["template_solution_code"])]

[template_valid(template,row["predictions"][0]) for template in row["template_solution_code"]]

[False, False]

In [170]:
print(single_patch(th_fil,row["predictions"][0]))


@@ -1 +1 @@

-\documentclass[tikz,border=2pt]{standalone}
+\documentclass[tikz,border=5]{standalone}
@@ -7 +7 @@

-\definecolor{§def(lust)}{rgb}{§range(0,0.6,0.13), §rangei(0.9,0.1), §range(0,0.6,0.13)}
+\definecolor{lust}{rgb}{0.0, 0.8, 0.0}


In [33]:
i

18

In [18]:
len(df)

86